특징 교차(feature cross) - 2개 이상의 개별 feature를 곱하거나 결합하여 새로운 합성 feature를 만드는 것
* x3 = x1 * x2
* goal = 데이터 간의 interaction을 모델에 직접 알려주기 위함

In [1]:
%pip install google-cloud-bigquery pyarrow pandas db-dtypes

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install google-cloud-bigquery-storage 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv


load_dotenv()

# 시스템 환경 변수에 구글 인증 키 경로 직접 등록
# 매직 명령어는 이 'GOOGLE_APPLICATION_CREDENTIALS'라는 이름을 찾아 인증합니다.
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('BIG_QUERY_KEY_PATH')

In [5]:
from google.cloud import bigquery

In [6]:
bq = bigquery.Client()
dataset = bigquery.Dataset(bq.dataset("babyweight"))

try:
  bq.create_dataset(dataset)
  print("Dataset created.")
except:
  print("Dataset already exists.")

Dataset already exists.


In [10]:
%load_ext google.cloud.bigquery

project_id = os.getenv('BIG_QUERY_PROJECT_ID')

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


In [11]:
%%bigquery --project $project_id

SELECT schema_name 
FROM `region-us`.INFORMATION_SCHEMA.SCHEMATA;

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,schema_name
0,babyweight
1,temp


In [12]:
%%bigquery --project $project_id
CREATE SCHEMA IF NOT EXISTS babyweight
OPTIONS(location="US");

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [13]:
%%bigquery --project $project_id
CREATE OR REPLACE TABLE
  babyweight.babyweight_data AS
SELECT
  weight_pounds,
  CAST(is_male AS STRING) AS is_male,
  mother_age,
  CASE
    WHEN plurality = 1 THEN "Single(1)"
    WHEN plurality = 2 THEN "Twins(2)"
    WHEN plurality = 3 THEN "Triplets(3)"
    WHEN plurality = 4 THEN "Quadruplets(4)"
    WHEN plurality = 5 THEN "Quintuplets(5)"
  END AS plurality,
  gestation_weeks,
  CAST(mother_race AS STRING) AS mother_race,
  FARM_FINGERPRINT(
    CONCAT(
      CAST(year AS STRING),
      CAST(month AS STRING)
    )
  ) AS hashmonth
FROM
  `bigquery-public-data.samples.natality`
WHERE
  year > 2000
  AND weight_pounds > 0
  AND mother_age > 0
  AND plurality > 0
  AND gestation_weeks > 0

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [14]:
%%bigquery --project $project_id
CREATE OR REPLACE TABLE
  babyweight.babyweight_data_train AS
SELECT
  weight_pounds,
  is_male,
  mother_age,
  plurality,
  gestation_weeks,
  mother_race
FROM
  babyweight.babyweight_data
WHERE
  ABS(MOD(hashmonth, 4)) < 3
  

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [15]:
%%bigquery --project $project_id
CREATE OR REPLACE TABLE
  babyweight.babyweight_data_eval AS
SELECT
  weight_pounds,
  is_male,
  mother_age,
  plurality,
  gestation_weeks,
  mother_race
FROM
  babyweight.babyweight_data
WHERE
  ABS(MOD(hashmonth, 4)) = 3
  

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [ ]:
%%bigquery --project $project_id
DROP MODEL IF EXISTS `babyweight.natality_model`;

In [18]:
%%bigquery --project $project_id
DROP MODEL IF EXISTS `babyweight.natality_model`;

/opt/homebrew/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [19]:
%%bigquery --project $project_id

CREATE OR REPLACE MODEL `babyweight.natality_model`
OPTIONS
  (MODEL_TYPE="DNN_REGRESSOR",
    HIDDEN_UNITS=[64, 32],
    INPUT_LABEL_COLS=["weight_pounds"],
    BATCH_SIZE=32,
    DATA_SPLIT_METHOD="NO_SPLIT") AS
SELECT
  weight_pounds,
  is_male,
  plurality,
  gestation_weeks,
  mother_age,
  CAST(mother_race AS string) AS mother_race
FROM
  babyweight.babyweight_data_train


Executing query with job ID: 0f15c89a-5303-43d1-ab07-30643bb08e8f
Query executing: 1.28s


ERROR:
 400 Concurrent model update with retrain is not supported.; reason: invalidQuery, location: query, message: Concurrent model update with retrain is not supported.

Location: US
Job ID: 0f15c89a-5303-43d1-ab07-30643bb08e8f



In [21]:
%%bigquery --project $project_id
SELECT * FROM `babyweight.INFORMATION_SCHEMA.MODELS`
WHERE model_name = 'natality_model';

Executing query with job ID: 5750cae8-6a3b-4225-9224-28e7ed0f5b9d
Query executing: 1.41s


ERROR:
 403 Access Denied: Table babyweight:INFORMATION_SCHEMA.MODELS: User does not have permission to query table babyweight:INFORMATION_SCHEMA.MODELS, or perhaps it does not exist.; reason: accessDenied, message: Access Denied: Table babyweight:INFORMATION_SCHEMA.MODELS: User does not have permission to query table babyweight:INFORMATION_SCHEMA.MODELS, or perhaps it does not exist.

Location: US
Job ID: 5750cae8-6a3b-4225-9224-28e7ed0f5b9d



In [22]:
query = """
SELECT
  *, SQRT(mean_squared_error) AS rmse
FROM
  ML.EVALUATE(MODEL `babyweight.natality_model`,
    (
    SELECT
      weight_pounds,
      is_male,
      plurality,
      gestation_weeks,
      mother_age,
      CAST(mother_race AS STRING) AS mother_race
    FROM
      babyweight.babyweight_data_eval ))
"""

In [23]:
df = bq.query(query).to_dataframe()
df.head()

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance,rmse
0,0.792551,1.03993,0.017722,0.648038,0.402351,0.403979,1.01977


빅쿼리에서 feature cross 사용해보기